# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rislantrs/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [4]:
import pandas as pd
!pip install catboost
from catboost import CatBoostClassifier

# 1. Ambil dataset yang sama dengan W06
url = "https://raw.githubusercontent.com/Rislantrs/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df_clean = df[df['avg_position'] > 0].copy()
df_clean['is_declining_label'] = (df_clean['trend_direction'] == 'down').astype(int)

# 2. Fitur yang divalidasi pada W06
features = ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']
X = df_clean[features]
y = df_clean['is_declining_label']

# 3. Fit model CatBoost
model = CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, verbose=0, random_seed=42)
model.fit(X, y)

# 4. Generate skor probabilitas penurunan (decline probability)
df_clean['pred_prob'] = model.predict_proba(X)[:, 1]

# 5. Logika pembuatan Action & Reason Code berdasarkan kondisi data
def assign_action_and_reason(row):
    # Hanya flag item dengan probabilitas penurunan tinggi (> 0.60)
    if row['pred_prob'] >= 0.60:
        if row['days_since_last_update'] > 90:
            return 'FULL_CONTENT_REFRESH', 'HIGH_DECAY_STALE'
        elif row['impressions_90d'] > 2000 and row['ctr'] < 0.03:
            return 'RESTRUCTURE_META_TITLE', 'LOW_CTR_HIGH_EXPOSURE'
        elif row['avg_position'] > 30:
            return 'EXPAND_CONTENT_INTENT', 'DEEP_SERP_VOLATILITY'
        else:
            return 'ROUTINE_EDITORIAL_AUDIT', 'ELEVATED_DECLINE_RISK'
    else:
        return 'MONITOR_ONLY', 'LOW_URGENCY'

df_clean[['recommended_action', 'reason_code']] = df_clean.apply(
    assign_action_and_reason, axis=1, result_type='expand'
)

# 6. Susun ranked queue (prioritas risiko tertinggi ke terendah)
ranked_queue = (
    df_clean[df_clean['recommended_action'] != 'MONITOR_ONLY']
    .sort_values(by='pred_prob', ascending=False)
    .reset_index(drop=True)
)

# Tampilkan ringkasan hasil
print(f"Total konten dianalisis: {len(df_clean)}")
print(f"Total antrean aksi (perlu tindakan): {len(ranked_queue)}")
ranked_queue[['content_id', 'pred_prob', 'recommended_action', 'reason_code', 'days_since_last_update', 'avg_position']].head(10)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00
Total konten dianalisis: 28795
Total antrean aksi (perlu tindakan): 13979


,content_id,pred_prob,recommended_action,reason_code,days_since_last_update,avg_position
0,content_9b28cf5ae4f3,0.952174,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,106,1.1
1,content_a37ce8ddc090,0.939253,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,106,2.3
2,content_7247c9f3c142,0.938205,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,104,0.2
3,content_b08562686d22,0.937787,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,106,2.0
4,content_2a57f2016a14,0.935608,ROUTINE_EDITORIAL_AUDIT,ELEVATED_DECLINE_RISK,15,0.6
5,content_b45048bf83d0,0.932238,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,104,0.9
6,content_d8020ed269ef,0.925181,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,104,1.2
7,content_d0351144852b,0.924106,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,104,0.6
8,content_97a08f7b106c,0.921960,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,104,1.5
9,content_0868e82484db,0.921566,FULL_CONTENT_REFRESH,HIGH_DECAY_STALE,106,4.1


### 1. Ranked Content Queue & Reason Codes

The objective of this queue is decision-support: prioritizing content units that exhibit the highest measured risk of traffic decline based on our audited CatBoost classifier. Rather than serving an opaque numerical probability alone, we prioritize actions by combining the model's decline probability (>= 0.60) with observable feature deficiencies.

Each item in the queue receives an explicit, human-readable reason code:
- `HIGH_DECAY_STALE`: High decline probability coupled with prolonged time since the last update (>90 days). Prime candidate for factual verification and date refresh (`FULL_CONTENT_REFRESH`).
- `LOW_CTR_HIGH_EXPOSURE`: Significant impression volume (>2,000 in 90 days) but measured low CTR (<3%). Signals a weak hook or snippet misalignment (`RESTRUCTURE_META_TITLE`).
- `DEEP_SERP_VOLATILITY`: Flagged risk where average ranking is deep (>30). Candidate for search intent realignment or content expansion (`EXPAND_CONTENT_INTENT`).
- `ELEVATED_DECLINE_RISK`: General high decline probability without a single extreme feature outlier, flagged for baseline review (`ROUTINE_EDITORIAL_AUDIT`).

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Cek distribusi tindakan yang direkomendasikan
action_counts = ranked_queue['recommended_action'].value_counts()
print("Action Distribution in Priority Queue:")
print(action_counts)

# Cek distribusi tindakan per client/domain (atau per archetype jika kolomnya ada)
group_col = 'archetype' if 'archetype' in ranked_queue.columns else 'client_id'

distribution_matrix = ranked_queue.groupby([group_col, 'recommended_action']).size().unstack(fill_value=0)
print(f"\nDistribution Matrix by {group_col} (Top 10):")
print(distribution_matrix.head(10))

# Batas operasional: verifikasi tidak ada client tunggal yang mendominasi antrean lebih dari 40%
client_concentration = ranked_queue['client_id'].value_counts(normalize=True).iloc[0]
print(f"\nMax client queue concentration: {client_concentration:.2%}")

Action Distribution in Priority Queue:
recommended_action
ROUTINE_EDITORIAL_AUDIT    7743
FULL_CONTENT_REFRESH       5379
EXPAND_CONTENT_INTENT       606
RESTRUCTURE_META_TITLE      251
Name: count, dtype: int64

Distribution Matrix by client_id (Top 10):
recommended_action  EXPAND_CONTENT_INTENT  FULL_CONTENT_REFRESH  \
client_id                                                         
client_02d20bbd7e                       0                    12   
client_0b918943df                       0                     2   
client_19581e27de                       5                  1944   
client_1a6562590e                       0                     3   
client_2c624232cd                       0                     0   
client_349c41201b                      52                     1   
client_3fdba35f04                      56                   769   
client_434c9b5ae5                       1                     0   
client_4e07408562                      72                   268   
client_

### 2. Intended Use and Operational Limits

**Intended Operational Use:**
- **Role:** This playbook serves strictly as a directional decision-support tool for editorial and SEO teams to prioritize manual content auditing.
- **Workflow Fit:** Instead of auditing all 28,795 content assets uniformly, teams focus effort on the top ~48.55% (13,979 pages) showing measured degradation signals.
- **Intervention Tiering:** Work is segmented into specific editorial buckets—triaging between routine audits (7,743 pages), factual/stale updates (5,379 pages), deep intent expansions (606 pages), and metadata/snippet adjustments (251 pages).

**Measured Operational Limits & Guardrails:**
- **Non-Causal Associations:** Decline probabilities ($P \ge 0.60$) reflect historical feature correlations from the CatBoost model, not causal certainty. A flagged page may be experiencing external market shifts rather than intrinsic content degradation.
- **Client Concentration Constraint:** The highest single-client share in the actionable queue is observed at 19.65% (`client_19581e27de`), well below our operational risk ceiling of 40%. The queue maintains balanced cross-domain utility.
- **Domain Scope:** Validated solely on informational search footprint across tracked client domains. The model is not calibrated for transactional checkout pages, gated assets, or breaking real-time news articles.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue['requires_deep_intent_review'] = ranked_queue['avg_position'] > 30

# Evaluasi berapa banyak antrean yang memerlukan verifikasi mendalam
review_breakdown = ranked_queue['requires_deep_intent_review'].value_counts()
print("=== HUMAN REVIEW AUDIT QUEUE ===")
print("True  = Butuh verifikasi manual mendalam (SERP > 30 / Rawan Volatilitas)")
print("False = Review editorial standar")
print("-" * 40)
print(review_breakdown)

# Tampilkan 5 contoh kasus yang paling wajib dicek manusia sebelum dieksekusi
deep_review_samples = ranked_queue[ranked_queue['requires_deep_intent_review']].head(5)
print("\nTop 5 Content Units Requiring Intent/SERP Verification:")
print(deep_review_samples[['content_id', 'pred_prob', 'avg_position', 'recommended_action', 'reason_code']])

=== HUMAN REVIEW AUDIT QUEUE ===
True  = Butuh verifikasi manual mendalam (SERP > 30 / Rawan Volatilitas)
False = Review editorial standar
----------------------------------------
requires_deep_intent_review
False    12752
True      1227
Name: count, dtype: int64

Top 5 Content Units Requiring Intent/SERP Verification:
              content_id  pred_prob  avg_position      recommended_action  \
21  content_2d6c50388f53   0.912946          47.4   EXPAND_CONTENT_INTENT   
42  content_424a5517925c   0.899818          36.9   EXPAND_CONTENT_INTENT   
48  content_dc55d7e3c008   0.898641          40.1  RESTRUCTURE_META_TITLE   
61  content_7c2bbd5f1b11   0.895424          39.7   EXPAND_CONTENT_INTENT   
66  content_21b3d827d451   0.893789          39.4  RESTRUCTURE_META_TITLE   

              reason_code  
21   DEEP_SERP_VOLATILITY  
42   DEEP_SERP_VOLATILITY  
48  LOW_CTR_HIGH_EXPOSURE  
61   DEEP_SERP_VOLATILITY  
66  LOW_CTR_HIGH_EXPOSURE  


### 3. Human Review Protocol & The Strict No-Go List

The CatBoost ranking acts strictly as a decision-support filter rather than an autonomous workflow. As measured in our Week-6 validation audit, pages ranking deep in search results (>30) are prone to false positives caused by search engine ranking volatility rather than true content decay. Human editorial review is mandatory before applying changes.

**Mandatory Human Review Checklist:**
1. **SERP Volatility Check (1,227 Priority Pages):** Exactly 1,227 content pages in the queue have an observed `avg_position > 30` (flagged under `requires_deep_intent_review`). Editors must verify whether ranking instability stems from broad query re-indexing before committing resources to deep rewrites (`EXPAND_CONTENT_INTENT`).
2. **Fact & Freshness Audit (5,379 Stale Pages):** For `FULL_CONTENT_REFRESH` actions, editors must manually check for outdated statistics, obsolete date references, and broken external links.
3. **Intent Alignment:** For `RESTRUCTURE_META_TITLE` items (251 pages), verify that rewrites improve click appeal without introducing clickbait that contradicts user search intent.

**The Strict No-Go List (Never Automate):**
- **Automated Text Generation & Direct CMS Publishing:** Generative AI must never automatically rewrite and push content live without human editorial sign-off.
- **Automated Content Pruning or Redirection:** Low-scoring or declining pages must never be auto-deleted or mass-redirected; historical link equity and brand footprint require human discretion.
- **Autonomous Overrides on High-Risk Content:** Content touching on regulated domains, compliance-critical topics, or top brand assets must not be modified based solely on model confidence scores.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Hitung dominasi kode alasan terbesar dalam antrean
reason_shares = ranked_queue['reason_code'].value_counts(normalize=True)
max_reason_share = reason_shares.iloc[0]
dominant_reason = reason_shares.index[0]

# 2. Cek apakah base rate prediksi antrean menyimpang drastis dari base rate W06 (~56.4%)
historical_base_rate = 0.5645
current_flagged_rate = len(ranked_queue) / len(df_clean)

# 3. Evaluasi ambang batas pemicu (Threshold Checks)
triggers = {
    "Dominant reason > 50%": max_reason_share > 0.50,
    "Queue density > 75% of total base": current_flagged_rate > 0.75,
    "Historical drift threshold reached": abs(current_flagged_rate - historical_base_rate) > 0.25
}

# 4. Ringkasan status monitoring
print("=== PLAYBOOK HEALTH & MONITORING AUDIT ===")
print(f"Top Reason Code:            {dominant_reason} ({max_reason_share:.2%})")
print(f"Current Queue Flagged Rate: {current_flagged_rate:.2%} (Historical Base Rate: {historical_base_rate:.2%})")
print("-" * 45)
print("Retrain Triggers Evaluation:")
for condition, triggered in triggers.items():
    status = "TRIGGERED (ALERT)" if triggered else "HEALTHY (PASS)"
    print(f" - {condition:<36}: {status}")

retrain_required = any(triggers.values())
print("-" * 45)
print(f"Immediate Retraining Required: {retrain_required}")

=== PLAYBOOK HEALTH & MONITORING AUDIT ===
Top Reason Code:            ELEVATED_DECLINE_RISK (55.39%)
Current Queue Flagged Rate: 48.55% (Historical Base Rate: 56.45%)
---------------------------------------------
Retrain Triggers Evaluation:
 - Dominant reason > 50%               : TRIGGERED (ALERT)
 - Queue density > 75% of total base   : HEALTHY (PASS)
 - Historical drift threshold reached  : HEALTHY (PASS)
---------------------------------------------
Immediate Retraining Required: True


### 4. Playbook Monitoring & Model Retrain Triggers

To prevent actionable recommendations from decaying due to search engine algorithm updates, market shifts, or seasonal swings, the health of this prioritization queue is tracked against explicit operational triggers.

**Observed Queue Health Metrics:**
- **Current Flagged Rate:** 48.55% of total evaluated content is marked for intervention, maintaining close alignment with our measured historical base rate of 56.45% (healthy baseline adherence).
- **Dominant Reason Concentration:** The generic risk category (`ELEVATED_DECLINE_RISK`) currently accounts for 55.39% of the actionable queue.

**Retrain Triggers & Current Operational Status:**
- **Trigger 1: Reason Code Saturation (>50% single code share) — [STATUS: TRIGGERED (ALERT)]**  
  Because `ELEVATED_DECLINE_RISK` represents 55.39% of the queue, the model displays broad risk bundling without strong feature differentiation for over half the flagged items. Immediate feature engineering or threshold recalibration is recommended.
- **Trigger 2: Queue Density Overload (>75% total base flagged) — [STATUS: HEALTHY (PASS)]**  
  At 48.55%, editorial capacity is preserved from bulk queue overflow.
- **Trigger 3: Historical Base Rate Drift (>25% divergence) — [STATUS: HEALTHY (PASS)]**  
  The observed rate remains within acceptable variance from the 56.45% baseline.

**Verdict & Action:**  
Retraining and parameter tuning are triggered immediately (`Immediate Retraining Required: True`) to disaggregate general decline risk into more distinct, granular operational actions.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json
import matplotlib.pyplot as plt

# 1. Pastikan direktori tujuan tersedia
outputs_dir = '../outputs'
figures_dir = '../figures'
os.makedirs(outputs_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

# 2. Ekspor Ranked Action Queue ke work/outputs/ (CSV ini diabaikan oleh Git via .gitignore)
queue_csv_path = os.path.join(outputs_dir, 'action_playbook_queue.csv')
ranked_queue.to_csv(queue_csv_path, index=False)
print(f"Exported queue CSV -> {queue_csv_path} ({len(ranked_queue)} rows)")

# 3. Ekspor Metrik Ringkasan ke JSON (sebagai bukti data paper)
playbook_metrics = {
    "total_evaluated_pages": int(len(df_clean)),
    "actionable_queue_size": int(len(ranked_queue)),
    "queue_percentage": float(len(ranked_queue) / len(df_clean)),
    "action_breakdown": ranked_queue['recommended_action'].value_counts().to_dict(),
    "reason_code_breakdown": ranked_queue['reason_code'].value_counts().to_dict(),
    "requires_intent_review_count": int(ranked_queue['requires_deep_intent_review'].sum()),
    "dominant_reason_share": float(max_reason_share)
}

metrics_json_path = os.path.join(outputs_dir, 'playbook_metrics.json')
with open(metrics_json_path, 'w') as f:
    json.dump(playbook_metrics, f, indent=2)
print(f"Exported metrics JSON -> {metrics_json_path}")

# 4. Buat dan simpan visualisasi distribusi aksi ke work/figures/
fig, ax = plt.subplots(figsize=(8, 4.5))
action_counts = ranked_queue['recommended_action'].value_counts()
action_counts.plot(kind='barh', ax=ax, color='#2c3e50', edgecolor='black')

ax.set_title('Distribution of Action Playbook Recommendations', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Flagged Content Pages', fontsize=10)
ax.set_ylabel('Recommended Action', fontsize=10)
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()

figure_path = os.path.join(figures_dir, 'action_distribution.png')
plt.savefig(figure_path, dpi=300)
plt.close()
print(f"Exported figure -> {figure_path}")

Exported queue CSV -> ../outputs/action_playbook_queue.csv (13979 rows)
Exported metrics JSON -> ../outputs/playbook_metrics.json
Exported figure -> ../figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.